# Financial Anomaly Detection: Prototype Comparison

This notebook is the runnable demonstration for the Python project. It imports the implementation from `src/` rather than duplicating it here. Use **Runtime → Run all** in Colab for the default no-secret, no-model-call walkthrough.

The three prototypes share the data retrieval and deterministic anomaly detector; they differ in how they produce explanations:

- **P1:** one baseline LLM explanation
- **P2:** SEC evidence retrieval followed by an LLM explanation
- **P3:** specialized agents followed by a critic

The cached AAPL evaluation is displayed first and requires no API calls. A live model run is optional. Results are preliminary: only one case was scored for each prototype.

## 1. Set up the project

The setup uses this checked-out project locally, or clones the public GitHub repository in Colab. For a one-click Colab run, commit and push the notebook, source, and evaluation snapshot first. The public-data sources used by the code are Yahoo Finance (via `yfinance`) and SEC Company Facts.

In [ ]:
%pip install -q pandas numpy yfinance requests python-dotenv google-genai

from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/trishekle/ECE-pitch.git"

PROJECT_DIR = Path.cwd()
while PROJECT_DIR != PROJECT_DIR.parent and not (PROJECT_DIR / "src").is_dir():
    PROJECT_DIR = PROJECT_DIR.parent

if not (PROJECT_DIR / "src").is_dir():
    PROJECT_DIR = Path("/content/ECE-pitch")
    if not (PROJECT_DIR / "src").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPOSITORY_URL, str(PROJECT_DIR)],
            check=True,
        )

if not (PROJECT_DIR / "src").is_dir():
    raise FileNotFoundError(f"Could not locate the project source under {PROJECT_DIR}")
sys.path.insert(0, str(PROJECT_DIR))
print(f"Project directory: {PROJECT_DIR}")
print("Public data sources: Yahoo Finance and SEC EDGAR Company Facts")

## 2. Load saved evaluation data

This notebook uses the compact, checked-in `notebooks/evaluation_snapshot.json`, containing the AAPL scores and measured latencies from the saved run. It avoids hidden local-file dependencies, uploads, and API calls in the default path. The full run artifacts remain in `outputs1.json` and `evaluation_results.json`.

In [ ]:
import json

SNAPSHOT_PATH = PROJECT_DIR / "notebooks" / "evaluation_snapshot.json"
if not SNAPSHOT_PATH.is_file():
    raise FileNotFoundError(f"Evaluation snapshot is missing: {SNAPSHOT_PATH}")

snapshot = json.loads(SNAPSHOT_PATH.read_text(encoding="utf-8"))
measurements = snapshot["measurements"]
case_info = snapshot["evaluation_case"]
print(f"Cached case: {case_info['case_id']} ({case_info['ticker']})")
print(f"Prototype runs: {case_info['prototype_runs']}; human-scored runs: {case_info['human_scored_runs']}")
print(snapshot["provenance"]["note"])

In [ ]:
import pandas as pd
from IPython.display import display

display(pd.DataFrame([{
    "Case ID": case_info["case_id"],
    "Ticker": case_info["ticker"],
    "Scored prototypes": case_info["human_scored_runs"],
    "Detected anomaly": case_info["detected_anomaly"],
}]))

## 3. Compare the saved scorecards and latency

Score ranges: anomaly correctness `0–1`; grounding, explanation quality, and limitations `0–2` each; unsupported claims are a count, so lower is better. The quality subtotal is the sum of grounding, explanation quality, and limitations (maximum `6`). These are cached measurements, not re-run by this notebook.

In [ ]:
score_rows = []
for row in measurements:
    score_rows.append({
        "Prototype": row["prototype"],
        "Cases scored": 1,
        "Anomaly correctness (0-1)": row["anomaly_correct"],
        "Grounding (0-2)": row["grounding"],
        "Explanation quality (0-2)": row["explanation_quality"],
        "Limitations (0-2)": row["limitations"],
        "Unsupported claims (count)": row["unsupported_claims"],
        "Quality subtotal (0-6)": row["grounding"] + row["explanation_quality"] + row["limitations"],
        "Latency (seconds)": row["latency_seconds"],
    })

scorecard = pd.DataFrame(score_rows)
display(scorecard.round(2))
print("N=1 case per prototype. This descriptive pilot cannot establish a general ranking.")

In [ ]:
prototype_rows = [
    {"Prototype": "P1", "Reasoning path": "Shared detector -> one LLM explanation", "Calls per run": 1},
    {"Prototype": "P2", "Reasoning path": "Shared detector -> SEC evidence retrieval -> one LLM explanation", "Calls per run": 1},
    {"Prototype": "P3", "Reasoning path": "Shared detector -> financial, filing, and market agents -> critic", "Calls per run": 4},
]
display(pd.DataFrame(prototype_rows))
print("Implementation: src/detection/anomalies.py and src/prototypes/p1_baseline.py, p2_retrieval.py, p3_multiagent.py")

## 4. Test-set status and interpretation

`tests/test_cases.json` documents intended synthetic test categories; the saved ticker-based AAPL run did not execute that synthetic dataset. They are shown as planned cases, not as test results. The current scored evaluation contains one AAPL case per prototype and no independent holdout set.

To reduce test-set leakage in a fuller study, freeze the thresholds and rubric before evaluating, keep a held-out set unseen during prompt/rule changes, score outputs without prototype labels where practical, and preserve failures. Here, the sample is too small to estimate general accuracy or false-positive rates. A financial anomaly is not evidence of fraud or wrongdoing.

In [ ]:
TEST_CASES_PATH = PROJECT_DIR / "tests" / "test_cases.json"
if TEST_CASES_PATH.is_file():
    test_cases = json.loads(TEST_CASES_PATH.read_text(encoding="utf-8"))
    display(pd.DataFrame([
        {
            "Test ID": case["id"],
            "Category": case["category"],
            "Expected detection": case["expected_detection"],
            "Executed in this saved comparison": False,
        }
        for case in test_cases
    ]))
else:
    print("No synthetic test-case definition file found.")

## 5. Critical assessment and Q&A prep

- **Research question:** Does added reasoning complexity improve analysis enough to justify extra latency and cost?
- **Fairness control:** P1/P2/P3 call the same deterministic detector and use the same ticker input; P2/P3 add evidence/reasoning steps, not new anomaly rules.
- **Preliminary result:** On the single scored AAPL case, P2 had grounding `2/2`, one unsupported claim, and 31.52 s latency; P3 had a higher explanation-quality score but three unsupported claims and 123.47 s latency. This is a trade-off observation, not a winner claim.
- **Limitations:** N=1 per prototype; subjective human rubric; no evaluated normal/negative case in the scored set; data-provider fields/periods may vary; market return and annual financial periods are only approximately aligned; no repeated trials, cost accounting, or confidence intervals.
- **Business case:** P2 may be a practical middle ground if its extra filing context reduces unsupported statements at acceptable latency/cost. A larger, blinded evaluation is needed before deployment.
- **Safety:** The system flags unusual metrics; it cannot establish intent, fraud, or misconduct. Explanations must separate facts, interpretations, hypotheses, and uncertainty.

## 6. Optional live demonstration (makes API calls)

Leave `RUN_LIVE = False` for the single-click cached walkthrough. Live inference requires an `AI_API_KEY` stored in Colab Secrets and access to the configured Gemini model; check the provider's current free-tier limits before use. Never paste or commit a key. Live data retrieval uses public Yahoo Finance and SEC endpoints. P3 makes four model calls and can take longer or fail with a 503. This optional section is not required to reproduce the cached scorecard.

In [ ]:
RUN_LIVE = False
LIVE_TICKER = "AAPL"
LIVE_CIK = "320193"

if not RUN_LIVE:
    print("Live run skipped. Set RUN_LIVE = True to make model/API calls.")
else:
    try:
        from google.colab import userdata
        api_key = userdata.get("AI_API_KEY")
    except Exception as exc:
        raise RuntimeError("Add AI_API_KEY to Colab Secrets before enabling the live run.") from exc

    if not api_key:
        raise RuntimeError("AI_API_KEY is missing from Colab Secrets.")

    import os
    import time
    os.environ["AI_API_KEY"] = api_key

    from src.prototypes.p1_baseline import run_p1
    from src.prototypes.p2_retrieval import run_p2
    from src.prototypes.p3_multiagent import run_p3

    live_functions = {
        "P1": lambda: run_p1(LIVE_TICKER),
        "P2": lambda: run_p2(LIVE_TICKER, LIVE_CIK),
        "P3": lambda: run_p3(LIVE_TICKER, LIVE_CIK),
    }
    live_results = []
    for prototype, function in live_functions.items():
        started = time.perf_counter()
        try:
            result = function()
            elapsed = round(time.perf_counter() - started, 3)
            live_results.append({
                "prototype": prototype,
                "ticker": LIVE_TICKER,
                "status": "success",
                "latency_seconds": elapsed,
                "output": result,
            })
            anomaly_data = result.get("anomalies", {})
            anomaly_list = anomaly_data.get("anomalies", []) if isinstance(anomaly_data, dict) else []
            print(f"{prototype}: success in {elapsed:.2f}s; {len(anomaly_list)} anomaly/anomalies")
            explanation = result.get("explanation") or result.get("final_report")
            if explanation:
                display(Markdown(f"### {prototype} live explanation\n\n{explanation}"))
        except Exception as exc:
            elapsed = round(time.perf_counter() - started, 3)
            error_code = getattr(exc, "code", None)
            live_results.append({
                "prototype": prototype,
                "ticker": LIVE_TICKER,
                "status": "failed",
                "latency_seconds": elapsed,
                "error_type": type(exc).__name__,
                "error_code": error_code,
            })
            print(f"{prototype}: failed after {elapsed:.2f}s ({type(exc).__name__}, code={error_code})")

    live_output_path = PROJECT_DIR / f"live_demo_results_{LIVE_TICKER.upper()}.json"
    live_output_path.write_text(json.dumps(live_results, indent=2, default=str), encoding="utf-8")
    display(pd.DataFrame([
        {key: row.get(key) for key in ("prototype", "ticker", "status", "latency_seconds", "error_type", "error_code")}
        for row in live_results
    ]))
    print(f"Saved live run summary to {live_output_path}")